In [ ]:

# GIVING THE TWO-STAGE PIPELINE ITS BEST HONEST CHANCE.
#
# The benchmark trained the pipeline's classifier on GROUND-TRUTH masks and then
# evaluated it on masks PREDICTED by the localiser. That is a train/test
# distribution mismatch, and it has never been tested. If the pipeline's deficit
# is caused by the mismatch rather than by mask quality, training on predicted
# masks should recover it. We also test two ways of preserving peri-lesional
# context that hard background-zeroing destroys.
#
# Kaggle traps guarded (both previously hit): P100 is sm_60, so pin cu121 torch
# BEFORE the first torch import and never os.execv; and strip the optional
# integrations whose callbacks raise at an epoch boundary.
import subprocess, sys, os, json, glob, time
info = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("GPU:", info or "NONE")
cap = info.split(",")[1].strip() if "," in info else ""
assert "torch" not in sys.modules
if cap.startswith("6."):
    print(f"compute capability {cap} is Pascal: pinning torch 2.5.1 + cu121")
    subprocess.run([sys.executable,"-m","pip","-q","install","torch==2.5.1","torchvision==0.20.1",
                    "--index-url","https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([sys.executable,"-m","pip","-q","uninstall","-y",
                "ray","wandb","comet_ml","mlflow","dvclive","neptune","clearml"], check=False)
subprocess.run([sys.executable,"-m","pip","-q","install","ultralytics==8.3.40","timm==1.0.11"], check=True)
import torch
print("torch", torch.__version__, "| arch", torch.cuda.get_arch_list())
assert torch.cuda.is_available()
_x=(torch.randn(64,64,device="cuda")@torch.randn(64,64,device="cuda")).sum().item(); torch.cuda.synchronize()
print("CUDA smoke test PASSED", round(_x,3))


In [ ]:

import numpy as np, cv2
from ultralytics import YOLO
UN = None
for dp, dns, fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("unmasked") and {"train","valid","test"} <= set(dns):
        UN = dp; break
assert UN, "unmasked classification dataset not found"
LOC = glob.glob("/kaggle/input/**/*1class_LEAKFREE.pt", recursive=True)[0]
print("unmasked root:", UN); print("localiser:", LOC)

m = YOLO(LOC)
OUT = "/kaggle/working/pred"
# Three mask regimes, all from the SAME predicted mask:
#   hard    - background set to 0, exactly what the deployed pipeline does
#   dilated - mask grown by 12 px, keeping a rim of peri-lesional skin
#   soft    - background attenuated to 25% rather than removed
REGIMES = ("hard","dilated","soft")
K = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25))
t0=time.time(); n=0
for split in ("train","valid","test"):
    for cls in sorted(os.listdir(f"{UN}/{split}")):
        d=f"{UN}/{split}/{cls}"
        if not os.path.isdir(d): continue
        for r_ in REGIMES: os.makedirs(f"{OUT}/{r_}/{split}/{cls}", exist_ok=True)
        for p in sorted(glob.glob(d+"/*")):
            img=cv2.imread(p)
            if img is None: continue
            H,W=img.shape[:2]
            res=m.predict(p, conf=0.05, verbose=False)[0]
            if res.masks is None or len(res.masks.data)==0:
                mk=np.ones((H,W),np.uint8)          # full-frame fallback, as deployed
            else:
                mm=res.masks.data.cpu().numpy().max(0)
                mk=(cv2.resize(mm,(W,H),interpolation=cv2.INTER_NEAREST)>0.5).astype(np.uint8)
            base=os.path.basename(p)
            cv2.imwrite(f"{OUT}/hard/{split}/{cls}/{base}", img*mk[...,None])
            dl=cv2.dilate(mk,K,iterations=1)
            cv2.imwrite(f"{OUT}/dilated/{split}/{cls}/{base}", img*dl[...,None])
            soft=(mk[...,None]+ (1-mk[...,None])*0.25)
            cv2.imwrite(f"{OUT}/soft/{split}/{cls}/{base}", (img*soft).astype(np.uint8))
            n+=1
            if n%400==0: print(f"  {n} images  ({(time.time()-t0)/60:.1f} min)", flush=True)
print(f"generated {n} images x {len(REGIMES)} regimes in {(time.time()-t0)/60:.1f} min")
for r_ in REGIMES:
    print(" ", r_, {s: len(glob.glob(f"{OUT}/{r_}/{s}/*/*")) for s in ("train","valid","test")})


In [ ]:

import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

IMG=224
tf_tr = transforms.Compose([transforms.Resize((IMG,IMG)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2), transforms.RandomRotation(10), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_ev = transforms.Compose([transforms.Resize((IMG,IMG)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

GT = None
for dp, dns, fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("masked") and {"train","valid","test"} <= set(dns) \
       and "unmasked" not in dp: GT = dp; break
print("ground-truth masked root:", GT)

def loaders(train_root, test_root, batch):
    tr=datasets.ImageFolder(f"{train_root}/train", tf_tr)
    va=datasets.ImageFolder(f"{train_root}/valid", tf_ev)
    te=datasets.ImageFolder(f"{test_root}/test",  tf_ev)
    cnt=np.bincount([y for _,y in tr.samples],minlength=3).astype(float)
    w=torch.tensor(cnt.sum()/(3*cnt),dtype=torch.float32).cuda()
    return (DataLoader(tr,batch_size=batch,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=64,num_workers=2),
            DataLoader(te,batch_size=64,num_workers=2), w)

def train_eval(train_root, test_root, seed, epochs=30, patience=7, batch=16, lr=1e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    tl,vl,el,w = loaders(train_root,test_root,batch)
    m=timm.create_model("swin_tiny_patch4_window7_224",pretrained=True,num_classes=3).cuda()
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.05)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    crit=nn.CrossEntropyLoss(weight=w); sc=torch.amp.GradScaler("cuda")
    best,bad,st=-1,0,None
    for ep in range(epochs):
        m.train()
        for x,y in tl:
            x,y=x.cuda(non_blocking=True),y.cuda(non_blocking=True); opt.zero_grad()
            with torch.amp.autocast("cuda"): loss=crit(m(x),y)
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step(); m.eval(); c=t=0
        with torch.no_grad():
            for x,y in vl:
                with torch.amp.autocast("cuda"): p=m(x.cuda()).argmax(1).cpu()
                c+=(p==y).sum().item(); t+=len(y)
        va=100*c/t
        if va>best: best,bad,st=va,0,{k:v.detach().clone() for k,v in m.state_dict().items()}
        else:
            bad+=1
            if bad>=patience: break
    m.load_state_dict(st); m.eval(); P=[];L=[]
    with torch.no_grad():
        for x,y in el:
            with torch.amp.autocast("cuda"): P+=m(x.cuda()).argmax(1).cpu().tolist()
            L+=y.tolist()
    acc=100*sum(int(a==b) for a,b in zip(P,L))/len(L)
    per={}
    for a,b in zip(P,L):
        per.setdefault(b,[0,0]); per[b][1]+=1
        if a==b: per[b][0]+=1
    return acc, 100*float(np.mean([c/t for c,t in per.values()])), best, P, L

PRED="/kaggle/working/pred"
# Regime A is the published configuration: trained on ground-truth masks,
# tested on predicted masks. B, C and D match the training distribution to the
# test distribution, which is what the deployed system actually sees.
ARMS = [
  ("A_gt_train__pred_test  (published)", GT,               f"{PRED}/hard"),
  ("B_pred_train__pred_test (matched)",  f"{PRED}/hard",   f"{PRED}/hard"),
  ("C_dilated_matched",                  f"{PRED}/dilated",f"{PRED}/dilated"),
  ("D_soft_matched",                     f"{PRED}/soft",   f"{PRED}/soft"),
]
SEEDS=[0,1,2]
res=[]
for name,trr,ter in ARMS:
    for s in SEEDS:
        t0=time.time(); a,b,v,P,L = train_eval(trr,ter,s)
        res.append({"arm":name,"seed":s,"acc":a,"bal":b,"val":v,"preds":P,"labels":L,
                    "minutes":round((time.time()-t0)/60,1)})
        json.dump(res, open("/kaggle/working/pipeline_best_chance.json","w"), indent=1)
        print(f"  {name:38} seed {s}: acc {a:6.2f}  bal {b:6.2f}  ({res[-1]['minutes']} min)", flush=True)

print("\n=== SUMMARY: pipeline accuracy on the 205 internal test images ===")
for name,_,_ in ARMS:
    A=np.array([r["acc"] for r in res if r["arm"]==name])
    B=np.array([r["bal"] for r in res if r["arm"]==name])
    print(f"  {name:38} {A.mean():6.2f} +- {A.std(ddof=1):4.2f}   balanced {B.mean():6.2f}")
print("\n  Standalone ConvNeXt-Large baseline over ten seeds: 83.61 +- 1.74")
print("  Published pipeline (regime A, ten seeds):          80.88 +- 1.50")
